# Лабораторный практикум по курсу «Синтез Речи», Университет ИТМО, 2026
**Лабораторная работа №1.** Нормализация текста

**Цель работы:** Определение ненормализованных текстовых данных и их обработка  
**Выполнила** Иванова Мария Кирилловна  
**Гркппа** М4121
**Преподаватель** Рыбин Сергей Витальевич

In [3]:
import os
import kagglehub
import pandas as pd
import csv
import re
import emoji
import librosa
import IPython.display as ipd
import librosa
from collections import Counter
from text_filter import TextFilter

from sklearn.metrics import confusion_matrix, classification_report


## 1. Загрузка и распаковка корпуса

In [2]:
# Локальный путь к распакованному датасету
base_path = "/mnt/storage/work_dir/databases/ruslan/ruslan_dataset"

# Пути к данным
ruslan_folder = os.path.join(base_path, "RUSLAN")
metadata_csv = os.path.join(base_path, "metadata_RUSLAN_22200.csv")

# Читаем метаданные
metadata = pd.read_csv(metadata_csv, sep='|', header=None)
metadata.columns = ['file_name', 'text']

print(f"Загружено {len(metadata)} записей")
print(f"Аудиофайлы в: {ruslan_folder}")

Загружено 22200 записей
Аудиофайлы в: /mnt/storage/work_dir/databases/ruslan/ruslan_dataset/RUSLAN


### Послушаем первый файл

In [4]:
first_file_name = metadata.iloc[0]['file_name']
first_file_path = os.path.join(ruslan_folder, first_file_name + '.wav')
first_text = metadata.iloc[0]['text']

audio, sr = librosa.load(first_file_path, sr=22050)

print(f"Текст: {first_text}")
print(f"Длительность: {len(audio)/sr:.2f} секунд")
ipd.display(ipd.Audio(audio, rate=sr))

Текст: С тревожным чувством берусь я за перо.
Длительность: 2.24 секунд


## 2. Разведочный анализ

### Статистика по набору

In [5]:
# Общее количество
total = len(metadata)
print(f" Всего текстов: {total}\n")

# Длина текстов (количество символов)
metadata['text_length'] = metadata['text'].str.len()
print(f"   \nДлина текста (в символах):")
print(f"   Минимальная: {metadata['text_length'].min()}")
print(f"   Максимальная: {metadata['text_length'].max()}")
print(f"   Средняя: {metadata['text_length'].mean():.1f}")
print(f"   Медианная: {metadata['text_length'].median():.1f}")

# Количество слов (приблизительно)
metadata['word_count'] = metadata['text'].str.split().str.len()
print(f" \nКоличество слов:")
print(f"   Минимальное: {metadata['word_count'].min()}")
print(f"   Максимальное: {metadata['word_count'].max()}")
print(f"   Среднее: {metadata['word_count'].mean():.1f}")
print(f"   Медианное: {metadata['word_count'].median():.1f}")

 Всего текстов: 22200

   
Длина текста (в символах):
   Минимальная: 10
   Максимальная: 723
   Средняя: 80.8
   Медианная: 85.0
 
Количество слов:
   Минимальное: 1
   Максимальное: 111
   Среднее: 12.0
   Медианное: 12.0


In [6]:
print("\nПримеры текстов")

for i in range(min(10, len(metadata))):
    print(f"{i+1:2d}. {metadata.iloc[i]['text']}")


Примеры текстов
 1. С тревожным чувством берусь я за перо.
 2. Кого интересуют признания литературного неудачника?
 3. Что поучительного в его исповеди?
 4. Да и жизнь моя лишена внешнего трагизма.
 5. Я абсолютно здоров.
 6. У меня есть любящая родня.
 7. Мне всегда готовы предоставить работу, которая обеспечит нормальное биологическое существование.
 8. Мало того, я обладаю преимуществами.
 9. Мне без труда удается располагать к себе людей.
10. Я совершил десятки поступков, уголовно наказуемых и оставшихся безнаказанными.


### 2.1 Цифры

In [ ]:
has_digits = metadata['text'].str.contains(r'\d', regex=True)
digit_examples = metadata[has_digits]

# Выводим статистику
print(f"Всего предложений с цифрами: {len(digit_examples)} из {len(metadata)} ({len(digit_examples)/len(metadata)*100:.2f}%)")

# Выводим все примеры
print("\nВсе предложения с цифрами:")
for i, (idx, row) in enumerate(digit_examples.iterrows(), 1):
    # Находим все цифры в тексте
    digits = re.findall(r'\d+', row['text'])
    print(f"{i}. {row['text']}\nцифры: {', '.join(digits)}\n")

Всего предложений с цифрами: 4 из 22200 (0.02%)

Все предложения с цифрами:
1. Брат разъезжал по отдаленным лагерным точкам. Ему предоставили казенную машину «ГАЗ‑61». 
цифры: 61

2. На Филиппинах кто‑то застрелил руководителя партийной оппозиции. Под Мелитополем разбился «ТУ‑129». 
цифры: 129

3. В сумочке ее лежало нечто, размером чуть поболее миниатюрного дамского браунинга «Элита‑16». 
цифры: 16

4. – Турбовинтовой МИ‑6, – заметил Пупс, вставая. – Е‑ё! – лениво крикнул он. Затем скрестил над головой руки. 
цифры: 6



### 2.2 Некириллические буквы

In [33]:
non_cyrillic = metadata['text'].str.contains(r'[A-Za-z]', regex=True)
non_cyrillic_examples = metadata[non_cyrillic]

# Выводим статистику
print(f"Всего предложений с латиницей: {len(non_cyrillic_examples)} из {len(metadata)} ({len(non_cyrillic_examples)/len(metadata)*100:.2f}%)")

# Выводим все примеры
if not non_cyrillic_examples.empty:
    print("\nВсе предложения с латиницей:")
    for i, (idx, row) in enumerate(non_cyrillic_examples.iterrows(), 1):
        # Находим все латинские буквы в тексте
        latin_letters = re.findall(r'[A-Za-z]+', row['text'])
        print(f"{i}. {row['text']}\nлатиница: {', '.join(latin_letters)}")

Всего предложений с латиницей: 0 из 22200 (0.00%)


### 2.3 Аббревиатуры

In [34]:
# Находим все слова в верхнем регистре
pattern = r'\b[А-Я]{3,}\b'

# Ищем предложения с капслоком
has_caps = metadata['text'].str.contains(pattern, regex=True, na=False)
caps_examples = metadata[has_caps]

# Собираем все слова в капсе
all_caps_words = []
for text in caps_examples['text']:
    all_caps_words.extend(re.findall(pattern, text))

unique_caps = set(all_caps_words)
freq_caps = Counter(all_caps_words)

# Статистика
print(f"Всего предложений с капслоком: {len(caps_examples)} из {len(metadata)} ({len(caps_examples)/len(metadata)*100:.2f}%)")
print(f"Уникальных слов в капсе: {len(unique_caps)}")

# Выводим все слова
print(f"\nВсе слова в капсе (3+ букв):")
print(', '.join(sorted(unique_caps)))

Всего предложений с капслоком: 261 из 22200 (1.18%)
Уникальных слов в капсе: 247

Все слова в капсе (3+ букв):
АБАНАМАТ, АЛЕКСАНДР, АМЕРИКАНСКИЙ, АМЕРИКИ, АНУК, АРИКА, АРМЯНСКУЮ, АТС, АХЧ, БАМ, БИЗНЕС, БОГОЛЮБОВ, БРЕЖНЕВ, БРЕЖНЕВУ, БРЕМЯ, БРУНО, БУР, ВАЛЕРИЙ, ВЕРТИКАЛЬНЫЙ, ВЕТРА, ВЛАДИМИР, ВЛАСТИ, ВМК, ВОЗВРАЩАЕМ, ВОЗВРАЩЕНИЕ, ВОСТОК, ВОХРА, ВСЕ, ВСТРЕТИЛИСЬ, ВСТРЕЧА, ВСТРЕЧУ, ГАЗ, ГАИ, ГАЛУШКО, ГАРЛЕМ, ГЕЙНЕ, ГЛЯЖУ, ГОРОД, ГПУ, ГРОЗНОЕ, ГРОЗОЙ, ГРОМ, ГРУСТНЫЙ, ГУЛАГ, ДАР, ДАРТАНЬЯНА, ДЕДУШКА, ДЕЛОМ, ДЕМОКРАТИИ, ДЕНЬГИ, ДЖОРДАНО, ДЖУНГЛЯХ, ДИРЕКТОР, ДИСТАНЦИЯ, ДЛЯ, ДОВЛАТОВ, ДОВЛАТОВА, ДОМА, ДОСААФ, ДОСТОЙНЫЕ, ДПИ, ДЯДЮШКА, ДЯР, ЕВРЕИ, ЕЛЕНА, ЕСТЬ, ЖЕРТВАМИ, ЖИТЬ, ЗАГАДОЧНЫЙ, ЗАРАБОТАТЬ, ЗДЕСЬ, ЗМЕЙ, ИЛЬВЕСОМ, ИНТЕРЕСНЫМ, КАК, КАПИТАЛА, КАРДИНАЛЕ, КВАСОМ, КВВК, КВН, КЛУБОК, КОНДРАШЕВ, КОНФЕРЕНЦИЯ, КОРИДОРЫ, КОТЕЛЬНИКОВ, КПП, КПСС, КПЭ, КРАСНОПЕРОВ, КРЕМЛЬ, КРИТИК, КРУГОМ, КТО, КУМПАН, КЭМ, ЛГУ, ЛЕОНИД, ЛЕЩЬ, ЛИНДА, ЛИНДЕ, ЛИРИЧЕСКОЕ, ЛИТЕРАТУРНЫЙ, ЛИТМО, ЛИТО, ЛОМО, ЛОРЕН, ЛОСП, ЛУЧШЕ,

In [ ]:
# Согласные буквы (русские)
consonants = 'БВГДЖЗЙКЛМНПРСТФХЦЧШЩ'
pattern = rf'\b[{consonants}]{{2,}}\b'  # 2+ согласных подряд

# Ищем предложения с согласными
has_consonants = metadata['text'].str.contains(pattern, regex=True, na=False)
consonant_examples = metadata[has_consonants]

# Список аббревиатур с гласными
extra_abbrev = ['АН', 'АТС', 'АХЧ', 'ГПУ', 'ДПИ', 'КПЭ', 'ЛГУ', 'МГУ', 
                'ПВО', 'МПВО', 'ПТУ', 'СНО', 'США', 'ТГУ', 'ТПИ', 'УВД', 'ЦО']

# Создаем паттерн для поиска этих аббревиатур
extra_pattern = '|'.join([rf'\b{abv}\b' for abv in extra_abbrev])

# Ищем предложения с дополнительными аббревиатурами
has_extra = metadata['text'].str.contains(extra_pattern, regex=True, na=False)

# Объединяем: предложения с согласными ИЛИ с дополнительными аббревиатурами
has_any_abbrev = has_consonants | has_extra
total_abbrev_examples = metadata[has_any_abbrev]

# Находим все аббревиатуры в этих предложениях (для статистики)
all_abbrev_found = []
for text in total_abbrev_examples['text']:
    # Ищем согласные
    found_consonants = re.findall(pattern, text)
    all_abbrev_found.extend(found_consonants)
    # Ищем дополнительные
    found_extra = re.findall(extra_pattern, text)
    all_abbrev_found.extend(found_extra)

unique_all = set(all_abbrev_found)
freq_all = Counter(all_abbrev_found)

print(f"Всего предложений с аббревиатурами: {len(total_abbrev_examples)} из {len(metadata)} ({len(total_abbrev_examples)/len(metadata)*100:.2f}%)")

print(f"\nУникальных сложночитаемых аббревиатур всего: {len(unique_all)}")

print(f"\nВсе найденные сложночитаемые аббревиатуры:")
print(', '.join(sorted(unique_all)))

print(f"\nТоп-10 самых частых аббревиатур:")
for word, count in freq_all.most_common(10):
    print(f"   {word}: {count} раз")

# Примеры предложений (только 5)
if not total_abbrev_examples.empty:
    print("\nПримеры предложений с аббревиатурами:")
    for i, (idx, row) in enumerate(total_abbrev_examples.head(5).iterrows(), 1):
        # Находим все аббревиатуры в этом предложении
        found = re.findall(pattern, row['text']) + re.findall(extra_pattern, row['text'])
        print(f"{i}. {row['text']}")
        print(f"  аббревиатуры: {', '.join(set(found))}\n")

Всего предложений с аббревиатурами: 59 из 22200 (0.27%)

Уникальных сложночитаемых аббревиатур всего: 34

Все найденные сложночитаемые аббревиатуры:
АН, АТС, АХЧ, БТ, ВМК, ГБ, ГПУ, ДПИ, КВВК, КВН, КП, КПП, КПСС, КПЭ, ЛГУ, МВД, МГУ, МПВО, МТС, НКВД, ПТУ, СНО, СС, ССР, СХШ, США, ТГУ, ТПИ, УВД, ФБР, ФД, ЦДЛ, ЦО, ЧК

Топ-10 самых частых аббревиатур:
   КПСС: 6 раз
   США: 4 раз
   УВД: 4 раз
   НКВД: 4 раз
   ЦДЛ: 4 раз
   КПЭ: 3 раз
   ЛГУ: 3 раз
   МГУ: 3 раз
   МТС: 2 раз
   ТГУ: 2 раз

Примеры предложений с аббревиатурами:
1. Отдел культуры и пропаганды цека КПСС
  аббревиатуры: КПСС

2. Отдел культуры Ленинградского ОК КПСС
  аббревиатуры: КПСС

3. Полковник отвечает Тут к нам в МТС прислали новенького.
  аббревиатуры: МТС

4. Ленинградского обкома КПСС
  аббревиатуры: КПСС

5. Несколько молодых преподавателей ТПИ уволили с работы.
  аббревиатуры: ТПИ



### 2.4 Сокращения

In [62]:
# паттерны для сокращений (включая варианты без точек)
patterns = {
    # Сокращения с точками (классические)
    'с_точками': r'\b(?:т\.\s*е|т\.\s*к|т\.\s*д|т\.\s*п|и\.\s*т\.\s*д|и\.\s*т\.\s*п|т\.\s*о)\b', 
    
    # Адресные сокращения (с точкой или без)
    'адреса': r'\b(?:ул\.|д\.|г-н|г-жа|г\.|пр\.|пер\.|пл\.|бульв\.|наб\.|ш\.|стр\.|рис\.|см\.|табл\.)',
    
    # Инициалы (одна или две буквы + точка + фамилия)
    'инициалы': r'\b[А-Я]\.\s*(?:[А-Я]\.\s*)?[А-Я][а-я]+\b',
    
    # Другие частые сокращения
    'прочие': r'\b(?:г-н|г-жа|тов\.|проф\.|доц\.|акад\.|д-р|канд\.)'
}

# Объединяем все паттерны
all_patterns = '|'.join(patterns.values())

# Ищем предложения с сокращениями
has_abbrev = metadata['text'].str.contains(all_patterns, regex=True, na=False)
abbrev_examples = metadata[has_abbrev]

# Собираем все сокращения
all_abbrev = []
for text in abbrev_examples['text']:
    all_abbrev.extend(re.findall(all_patterns, text))

unique_abbrev = set(all_abbrev)
freq_abbrev = Counter(all_abbrev)

print(f"Всего предложений с сокращениями: {len(abbrev_examples)} из {len(metadata)} ({len(abbrev_examples)/len(metadata)*100:.2f}%)")
print(f"Уникальных сокращений: {len(unique_abbrev)}")

print(f"\nВсе найденные сокращения:")
print(', '.join(sorted(unique_abbrev)))

print(f"\nЧастота сокращений (топ-15):")
for word, count in freq_abbrev.most_common(15):
    print(f"   {word}: {count} раз")

# Примеры предложений
if not abbrev_examples.empty:
    print("\n Примеры предложений с сокращениями (первые 10):")
    for i, (idx, row) in enumerate(abbrev_examples.head(10).iterrows(), 1):
        found = re.findall(all_patterns, row['text'])
        print(f"{i}. {row['text']}")
        print(f"   → сокращения: {', '.join(found)}\n")

Всего предложений с сокращениями: 49 из 22200 (0.22%)
Уникальных сокращений: 52

Все найденные сокращения:
А. Беспристрастный, А. Горелова, А. Иванов, А. Мельдером, А. П. Керн, А. Розена, А. С. Пушкина, А. С. Скачинский, Б. Нейфах, В. Беззубовым, В. Котельников, В. Котельникову, В. Сильд, В. Смирнов, В. Щербаков, В.И.Ленин, Г. Друкер, Г. Туронок, Е. Эткинда, И. Бродский, И. Гаспль, И. Популовский, К. В. Успенского, К. Малышев, Л. Кокк, Л. Я. Гинзбург, Л.Агаповой, Л.Юргенс, М. Зощенко, М. Обработала, М. Он, Н. Е. Буренина, Н. Предлагаю, Н. Сагаловского, Н.Сагаловского, Р. Сийрак, Р. Хемингуэй, С. Довлатов, С. Довлатова, С.Довлатова, Ф. Кривина, Х. Ныммисте, Э. Кураева, Э. Л. Буш, Э.Ильвес, Э.Л.Буш, Я. Беглые, Я. Р. Сначала, т. д, т. е, тов., ул.

Частота сокращений (топ-15):
   Г. Друкер: 3 раз
   И. Популовский: 2 раз
   Г. Туронок: 2 раз
   С. Довлатов: 2 раз
   т. д: 2 раз
   В. Щербаков: 1 раз
   В. Смирнов: 1 раз
   Н. Предлагаю: 1 раз
   Е. Эткинда: 1 раз
   К. Малышев: 1 раз
   Б

In [63]:
# Паттерн для найденных сокращений: г, д, т. д, т. е, ул
pattern = r'\b(?:тов\.|д\.|т\.\s*д|т\.\s*е|ул\.)\.?'

# Ищем предложения
has_abbrev = metadata['text'].str.contains(pattern, regex=True, na=False)
abbrev_examples = metadata[has_abbrev]

print(f"Найдено предложений: {len(abbrev_examples)}")

# Прослушиваем первые 5
for i, (idx, row) in enumerate(abbrev_examples.head(5).iterrows(), 1):
    file_path = os.path.join(ruslan_folder, row['file_name'] + '.wav')
    if os.path.exists(file_path):
        audio, sr = librosa.load(file_path, sr=22050)
        print(f"\n{i}. {row['text']}")
        ipd.display(ipd.Audio(audio, rate=sr))

Найдено предложений: 4

1. В. Котельников намеревался показать их своему родственнику, главному редактору Кинокомитета тов.



2. …Контора размещалась тогда на улице Пикк. Строго напротив здания Госбезопасности (ул. Пагари, один). 



3. Они – моя неврастения, злость, апломб, беспечность. И т. д. И самая кровавая война – бой призраков. 



4. Я вынужден признать, что кто‑то из нас может стать первой жертвой этого критерия, но от этого он не становится менее надежным, а главное – менее единственным, если так можно говорить по‑русски… Мы уже говорили две литературы или одна – это, по сути дела, разговор о будущем литературы, эта литература едина, в ней есть элементы, которые присущи любой здоровой культуре, т. е. сосуществуют – реалистическая проза, авангард, наличествует сатира, нигилизм и т. д. В заключение я хочу сказать, что будущее литературы лежит, мне кажется, в сфере осознания литературой собственных прав и собственных возможностей.


Следующие сокращения читаются:
1. Ул -> 'ул'  
2. т. е. -> 'то есть'  
3. т. д. -> 'т. д.'
4. тов. -> 'тов.'

### 2.5 Технические символы, необычные знаки и обозначения

In [69]:
# Стандартные знаки (включая все виды тире и дефисов)
standard_punct = r'.,!?;:…—\-–‑"\'«»()'

# Технические символы и обозначения (добавляем в поиск)
#tech_and_symbols = r'[*\/@+<>{}\[\]\\|~`=^&]%°$€£¥©®™№±√∫∑∞≈≠≤≥'

# Нестандартные знаки - все, что не входит в стандартные + техсимволы + обозначения
# Сначала находим все, что не является стандартным знаком
nonstandard_punct = r'[^\w\s' + standard_punct + r']'

# Ищем предложения с нестандартными знаками (включая техсимволы и обозначения)
has_nonstandard = metadata['text'].str.contains(nonstandard_punct, regex=True, na=False)
nonstandard_examples = metadata[has_nonstandard]

# Собираем все нестандартные знаки
all_chars = []
for text in nonstandard_examples['text']:
    all_chars.extend(re.findall(nonstandard_punct, text))

unique_chars = set(all_chars)
char_freq = Counter(all_chars)

print(f"Всего предложений с нестандартными знаками и техническими символами: {len(nonstandard_examples)} из {len(metadata)} ({len(nonstandard_examples)/len(metadata)*100:.2f}%)")
print(f"Уникальных нестандартных знаков: {len(unique_chars)}")

if unique_chars:
    print(f"\nВсе нестандартные знаки:")
    print(', '.join(sorted(unique_chars)))

    print(f"\nЧастота нестандартных знаков:")
    for char, count in char_freq.most_common():
        print(f"   '{char}': {count} раз")

# Примеры предложений
if not nonstandard_examples.empty:
    print("\n Примеры предложений с нестандартными знаками (первые 5):")
    for i, (idx, row) in enumerate(nonstandard_examples.head(5).iterrows(), 1):
        found = re.findall(nonstandard_punct, row['text'])
        print(f"{i}. {row['text']}")
        if found:
            print(f" нестандартные знаки: {', '.join(found)}\n")

Всего предложений с нестандартными знаками и техническими символами: 70 из 22200 (0.32%)
Уникальных нестандартных знаков: 9

Все нестандартные знаки:
*, /, <, >, ́, ’, “, ”, „

Частота нестандартных знаков:
   '„': 72 раз
   '“': 57 раз
   '”': 13 раз
   '’': 6 раз
   '/': 4 раз
   '́': 4 раз
   '*': 1 раз
   '<': 1 раз
   '>': 1 раз

 Примеры предложений с нестандартными знаками (первые 5):
1. Мечтаем получить от вас рецензию.”
 нестандартные знаки: ”

2. Я на букву О /библиография к Окуджаве/.
 нестандартные знаки: /, /

3. А Лев Уфлянд* еще больше подливает желчи, плюет на русский народ.
 нестандартные знаки: *

4. /Например, вертухай, как вы соизволили дружески меня поименовать.
 нестандартные знаки: /

5. Единственная драма этого автора не имела театральной биографии, оставаясь „повестью в диалогах”. 
 нестандартные знаки: „, ”



### 2.6 Эмодзи

In [65]:
# Текстовые смайлы
text_emojis_pattern = r'[:;=][-o*^]?[()\[\]{}<>DdpPрР/\\|3cCOo0]|[:;=][()]|[XO][-]?[()]'

def extract_all_emojis(text):
    # Юникод-эмодзи через библиотеку
    unicode_emojis = [char for char in text if emoji.is_emoji(char)]
    # Текстовые смайлики через regex
    text_emojis = re.findall(text_emojis_pattern, text)
    return unicode_emojis + text_emojis

def has_any_emoji(text):
    # Юникод-эмодзи
    if any(emoji.is_emoji(char) for char in text):
        return True
    # Текстовые смайлики
    if re.search(text_emojis_pattern, text):
        return True
    return False

# Поиск эмодзи
has_emoji_flag = metadata['text'].apply(has_any_emoji)
emoji_examples = metadata[has_emoji_flag]

all_emojis = []
for text in emoji_examples['text']:
    all_emojis.extend(extract_all_emojis(text))

unique_emojis = set(all_emojis)
freq_emojis = Counter(all_emojis)

# Статистика
print(f"Всего предложений с эмодзи: {len(emoji_examples)} из {len(metadata)} ({len(emoji_examples)/len(metadata)*100:.2f}%)")
print(f"Уникальных эмодзи: {len(unique_emojis)}")

if unique_emojis:
    print(f"\nВсе найденные эмодзи:")
    print(', '.join(sorted(unique_emojis)))

# Примеры
if not emoji_examples.empty:
    print("\nПримеры предложений с эмодзи (первые 5):")
    for i, (idx, row) in enumerate(emoji_examples.head(5).iterrows(), 1):
        found = extract_all_emojis(row['text'])
        print(f"{i}. {row['text']}")
        if found:
            print(f"   → эмодзи: {', '.join(set(found))}\n")

Всего предложений с эмодзи: 0 из 22200 (0.00%)
Уникальных эмодзи: 0


### 2.7 Междометия

In [48]:
short_pattern = r'\b(?:ага|агу|ай|ахаха|ах|брр|гм|гмм|ммм|е|ё|ого|ой|ох|охохо|трах|угу|ура|ух|ха|хм|хмм|хы|ых|ыых|э|эх|ю|мда|хо|хе|хи|гы|гх|фу|фи|тьфу|упс|угу|эй|эге|эхе|эх)\b'
e_pattern = r'\b[еёэы]\b|\b[еёэы]\s*[‒–—‐-]\s*[еёэ](?:\s*[‒–—‐-]\s*[еёэ])?\b'
interjection = r'\b(?:х[еёэ]х[еёэ])+\b|\b(?:хаха)+\b|\b(?:хихи)+\b|\b(?:хохо)+\b|\b[ы]{2,}\b'
#interjection = r'\b(?:х[еёэ]х[еёэ])+\b|\b(?:хаха)+\b|\b(?:хихи)+\b|\b(?:хохо)+\b|\b[ы]{2,}\b|\b(?:ха|хе|хи|хо|хы|га|го|гу|ой|ай|эй|ух|ах|ох)(?:\s*[‒–—‐-]\s*(?:ха|хе|хи|хо|хы|га|го|гу|ой|ай|эй|ух|ах|ох))+\b'

combined_pattern = f'{short_pattern}|{e_pattern}|{interjection}'

# Убираем инициалы из текста перед поиском
metadata['text_clean'] = metadata['text'].apply(
    lambda t: re.sub(r'\b[А-Я]\.\s*(?=[А-Я])', '', t)
)

# Ищем предложения с междометиями
has_interj = metadata['text_clean'].str.contains(combined_pattern, regex=True, case=False, na=False)
interj_examples = metadata[has_interj]

print(f"Найдено предложений с междометиями: {len(interj_examples)} из {len(metadata)} ({len(interj_examples)/len(metadata)*100:.2f}%)")

if not interj_examples.empty:
    print("\nПримеры предложений с междометиями (первые 5):")
    for i, (idx, row) in enumerate(interj_examples.head(5).iterrows(), 1):
        # Находим сами междометия в тексте
        found = re.findall(combined_pattern, row['text_clean'])
        # Очищаем от пустых значений
        found_clean = [f for f in found if f]
        print(f"{i}. {row['text_clean']}")
else:
    print("Междометий не найдено в корпусе.")

Найдено предложений с междометиями: 104 из 22200 (0.47%)

Примеры предложений с междометиями (первые 5):
1. Ага, подумают, возомнил себя непризнанным гением!
2. Ах, не прикасайтесь ко мне!
3. Ах, рассказы?
4. Ах, в отпуске?
5. Вы говорите Ах, какая миленькая собачка!  И быстрым движением щелкаете ее по носу.


### 2.7.1 Повторяющиеся знаки препинания

In [5]:
# Паттерны для поиска
patterns_debug = {
    'special_punct': r'[!?]{2,}|[!?][,.?!]|[,.]?[!?]{2,}|[()\[\]{}]',

}

# Проверяем каждый паттерн
for name, pattern in patterns_debug.items():
    has_pattern = metadata['text'].str.contains(pattern, regex=True, na=False)
    examples = metadata[has_pattern]

    # Процент от всего корпуса
    percent = len(examples) / len(metadata) * 100

    print(f"Найдено: {len(examples)} предложений из {len(metadata)} ({percent:.2f}%)")
    
    if not examples.empty:
        print(f"\nПримеры (первые 5):")
        for i, (idx, row) in enumerate(examples.head(5).iterrows(), 1):
            # Находим сами символы в тексте
            found = re.findall(pattern, row['text'])
            print(f"{i}. {row['text']}")
            print(f"  найдено: {', '.join(found)}\n")
    else:
        print("   (не найдено)")

Найдено: 1242 предложений из 22200 (5.59%)

Примеры (первые 5):
1. За что же моя рядовая, честная, единственная склонность подавляется бесчисленными органами, лицами, институтами великого государства??
  найдено: ??

2. Вы страшное говно, мон колонель, не обессудьте!..
  найдено: !.

3. Какать в одном поле не сяду!..
  найдено: !.

4. Неужели какать рядом с вами такая уж большая честь?!
  найдено: ?!

5. Да это же писатель!..
  найдено: !.



### 2.7.2 То же самое, но без комбинации '?!'

In [70]:
# Паттерны для поиска
patterns_debug = {
    'special_punct': r'[!?]{2,}|[!?][,.?!]|[,.]?[!?]{2,}|[()\[\]{}]',
}

# Проверяем каждый паттерн
for name, pattern in patterns_debug.items():
    # Сначала находим все предложения с паттерном
    has_pattern = metadata['text'].str.contains(pattern, regex=True, na=False)
    examples = metadata[has_pattern]
    
    # Исключаем те, где есть "?!" (считаем их нормализованными)
    has_exception = examples['text'].str.contains(r'\?\!', regex=True, na=False)
    examples_filtered = examples[~has_exception]

    # Процент от всего корпуса
    percent = len(examples_filtered) / len(metadata) * 100

    print(f"Найдено: {len(examples_filtered)} предложений из {len(metadata)} ({percent:.2f}%) с несколькими знаками препинания подряд, исключая ?!")
    
    if not examples_filtered.empty:
        print(f"\nПримеры (первые 5):")
        for i, (idx, row) in enumerate(examples_filtered.head(5).iterrows(), 1):
            found = re.findall(pattern, row['text'])
            # Убираем из найденного "?!"
            found_filtered = [f for f in found if f != '?!']
            print(f"{i}. {row['text']}")
            if found_filtered:
                print(f"  найдено: {', '.join(found_filtered)}\n")
            else:
                print(f"  (только '?!', исключено)\n")
    else:
        print("   (не найдено)")

Найдено: 874 предложений из 22200 (3.94%) с несколькими знаками препинания подряд, исключая ?!

Примеры (первые 5):
1. За что же моя рядовая, честная, единственная склонность подавляется бесчисленными органами, лицами, институтами великого государства??
  найдено: ??

2. Вы страшное говно, мон колонель, не обессудьте!..
  найдено: !.

3. Какать в одном поле не сяду!..
  найдено: !.

4. Да это же писатель!..
  найдено: !.

5. Расстреливать надо таких писателей!..
  найдено: !.



## Результаты разведочного анализа корпуса RUSLAN

### 1. Общая статистика
- **Всего фраз:** 22 200
- **Длина текста:** от 10 до 723 символов (в среднем ~81 символ, ~12 слов)
- **Аудио:** 22 200 файлов `.wav`, 22050 Гц

---

### 2. Типы ненормализованных данных и стратегия нормализации

| Тип | Найдено | % от корпуса | Примеры | Возможная нормализация |
|-----|---------|--------------|---------|------------------------|
| Числа | 4 | 0.02% | `ГАЗ‑61`, `ТУ‑129`, `Элита‑16` | Удалять целиком предложение, поскольку при замене на буквенное прочтение важен еще и падеж|
| Латиница | 0 | 0.00% | — | Отсутствует |
| Аббревиатуры (сложночитаемые) | 59 | 0.27% | `КПСС`, `МГУ`, `США`, `НКВД`, `Минюст` | Аббревиатуры, читающиеся словом (например, "ГУЛАГ" оставлять нетронутыми) |
| Сокращения (инициалы, адреса) | 52 | 0.23% | `А.С. Пушкин`, `ул.`, `г-н` | Инициалы удалить, стандартные сокращения не меняющие форму слова заменить (`т.е.`→`то есть`) за исключением тех, что диктор читает как написано (т.д., ул.) |
| Технические символы, обозначения, нестандартные кавычки/апострофы | 70 | 0.32% | `*`, `/`, `<`, `>` | Удалить символы, а сам текст оставить, нестандартные кавычки заменить стандартными |
| Повторяющиеся знаки препинания | 1242 (включая комбинацию `?!`), 874 (исключая `?!`) | 3.94% (без `?!`), 5.59% (с `?!`) | `!!!`, `?!.`, `))` | Заменить одним знаком. В приоритете `?`, затем `!`|
| Междометия | 104 | 0.47% | `ага`, `ой`, `мда`, `хехе`, `Ы-ы` | Удалять полностью, поскольку удаление только междометий может затронуть смысл фразы (например: `Митрофанов вдруг напрягся. – Ы‑ы‑а, – проговорил он. – Что? – спросила моя жена.` ) |
| Эмодзи | 0 | 0.00% | — | Отсутствуют, их можно удалять без потери смысла предложения |


### 3. Выводы и стратегия нормализации

**Корпус достаточно чистый ( более 94% нормализованных фраз).**   
Основные проблемы:
1. **Технические символы** (0.32%)  - удаляются без потери смысла.
2. **Повторяющиеся знаки препинания** (от 3.94% до 5.59%) - сводятся к одному знаку.
3. **Аббревиатуры и сокращения** (0.5%) - требуют словаря, поэтому сложночитаемые аббревиатуры проще удалить с предложением, оставляя лишь те, что читаются как обычные слова.
4. **Междометия** (0.47%) - зачастую важны для понимания смысла и эмоциональной окраски сказанного, просто удалить нельзя.

## 3. Фильтр нормализованных текстов

In [4]:
# Загружаем dev-набор
dev_path = "data/dev_sentences.csv"
dev_files = pd.read_csv(dev_path, sep="|", encoding="utf-8", header=0)

# Создаём фильтр
filter = TextFilter()

# Применяем фильтр
dev_files['predicted'] = dev_files['text'].apply(filter.filter)

# Сравнение с разметкой
print("\nМатрица ошибок (confusion matrix):")
cm = confusion_matrix(dev_files['is_normalized'], dev_files['predicted'])
print(cm)

print("\nОтчёт по классификации:")
print(classification_report(dev_files['is_normalized'], dev_files['predicted']))


Матрица ошибок (confusion matrix):
[[143  13]
 [  2 142]]

Отчёт по классификации:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95       156
           1       0.92      0.99      0.95       144

    accuracy                           0.95       300
   macro avg       0.95      0.95      0.95       300
weighted avg       0.95      0.95      0.95       300



### Результат фильтрации
#### Корпус данных достаточно чистый. При запуске фильтра лишь небольшой процент текстов был выделен как ненормализованный. F1-score достиг 95%. Ложных срабатываний 13, пропущенных 2 (это предложения с комбинацией `?!.`, которую фильтр оставляет как нормализованную). Среди ложных срабатываний встречаются примеры с необработанными пробелами. Такие случаи правильнее исправлять в нормализаторе, а не отбрасывать фильтром.

## 4. Нормализатор текста

### Виды нормализации:
1. Удаление лишних пробелов;
2. Замена нестандартных символов, удаление необязательных знаков и технических символов, не влияющих на смысловую часть;
3. Удаление смайликов;
4. Сведение комбинаций из знаков препинания к одному доминирующему знаку;
5. Раскрытие сокращений там, где это возможно и соответствует озвучке текста.

### Необрабатываемые ситуации:
1. Междометия (важны в контексте);
2. Числительные (необходимо учитывать падеж и фору слова);
3. Аббревиатуры (необходим словарь аббревиатур и дообучение на нем, можно оставить только те, что читаются как полноценное слово и убирать предложения со сложночитаемыми аббревиатурами);
4. Латиница не встречалась в данном наборе. В целом, предложения с ней можно удалить, поскольку подобные слова редки.
5. Обработка инициалов. Их было бы вполне логично убрать из текста, поскольку люди чассто их просто пропускают, а зачитывают только фамилию. Однако важно соответствие текста аудио, а в аудио диктор озвучивает побуквенно инициалы. Поэтому предложения с ними не удастся сохранить, не выискивая конкретных людей вручную или в соответствующей базе.

### Опробация на примерах

In [5]:
from text_normalizer import TextNormalizer

normalizer = TextNormalizer()

test_texts = [
    "ул. Ленина, т. е. Сталина",
    "Привет!!! Как дела?!.",
    "текст * мусор / лишнее",
    "т.е. я пришёл",
    "К сожалению, я не расслышал ваш „ответ”."
]

for t in test_texts:
    print(f"Было: {t}")
    print(f"Стало: {normalizer.normalize(t)}\n")

Было: ул. Ленина, т. е. Сталина
Стало: ул. Ленина, то есть Сталина

Было: Привет!!! Как дела?!.
Стало: Привет! Как дела?

Было: текст * мусор / лишнее
Стало: текст мусор лишнее

Было: т.е. я пришёл
Стало: то есть я пришёл

Было: К сожалению, я не расслышал ваш „ответ”.
Стало: К сожалению, я не расслышал ваш "ответ".



## 5. Фильтрация корпуса RUSLAN

In [4]:
from text_filter import TextFilter
from text_normalizer import TextNormalizer

# Загружаем корпус
base_path = "/mnt/storage/work_dir/databases/ruslan/ruslan_dataset"
metadata_csv = os.path.join(base_path, "metadata_RUSLAN_22200.csv")

metadata = pd.read_csv(metadata_csv, sep='|', header=None)
metadata.columns = ['file_name', 'text']

print(f"Загружено {len(metadata)} записей")

# Создаём фильтр и нормализатор
filter = TextFilter()
normalizer = TextNormalizer()

# Применяем фильтр к исходному тексту (для статистики)
metadata['filtered'] = metadata['text'].apply(filter.filter)

# Применяем нормализатор ко всем текстам
metadata['normalized'] = metadata['text'].apply(normalizer.normalize)

# Применяем фильтр к нормализованному тексту
metadata['filtered_after_norm'] = metadata['normalized'].apply(filter.filter)

total = len(metadata)

# Исходные
original_normalized = (metadata['filtered'] == 1).sum()
original_bad = (metadata['filtered'] == 0).sum()

# После нормализации
after_norm_normalized = (metadata['filtered_after_norm'] == 1).sum()
after_norm_bad = (metadata['filtered_after_norm'] == 0).sum()

# Сколько удалось спасти
saved = after_norm_normalized - original_normalized

print(f"\nИсходный текст (до нормализации):")
print(f"   Нормализованных: {original_normalized} ({original_normalized/total*100:.2f}%)")
print(f"   Ненормализованных: {original_bad} ({original_bad/total*100:.2f}%)")

print(f"\nПосле нормализации:")
print(f"   Нормализованных: {after_norm_normalized} ({after_norm_normalized/total*100:.2f}%)")
print(f"   Ненормализованных: {after_norm_bad} ({after_norm_bad/total*100:.2f}%)")

print(f"\nУдалось нормализовать (исправить): {saved} предложений")
print(f"Остались ненормализованными: {after_norm_bad} предложений")

# Оставляем только предложения, прошедшие фильтр после нормализации
clean = metadata[metadata['filtered_after_norm'] == 1].copy()

OUTPUT_PATH = "../../data/metadata_RUSLAN_22200_normalized.csv"
CSV_KWARGS = {"sep": "|", "quoting": csv.QUOTE_NONE}

clean[['file_name', 'text', 'normalized']].to_csv(
    OUTPUT_PATH, index=False, header=False, **CSV_KWARGS
)

Загружено 22200 записей

Исходный текст (до нормализации):
   Нормализованных: 20664 (93.08%)
   Ненормализованных: 1536 (6.92%)

После нормализации:
   Нормализованных: 21923 (98.75%)
   Ненормализованных: 277 (1.25%)

Удалось нормализовать (исправить): 1259 предложений
Остались ненормализованными: 277 предложений


## Результат обработки корпуса RUSLAN

### Итоги нормализации

| Этап | Нормализованных | Ненормализованных |
|------|-----------------|-------------------|
| **До нормализации** | 20664 (93.08%) | 1536 (6.92%) |
| **После нормализации** | 21923 (98.75%) | 277 (1.25%) |

**Нормализатор исправил 1 259 предложений** (5.67% корпуса), которые фильтр без него удалил бы.

### Вывод

Корпус RUSLAN **сохранил 98.75% объёма** после фильтрации и нормализации, потеряно всего **277 предложений (1.25%)**. Ненормализованными остались только те случаи, которые **нельзя исправить без потери смысла**: сложночитаемые аббревиатуры (`КПСС`, `НКВД`), инициалы(поскольку в аудио они озвучиваются по одной букве), междометия (`ага`, `мда`) и некоторые цифровые обозначения (`ГАЗ-61`).

Без нормализатора фильтр отбросил бы **1536 предложений**, но благодаря предварительной обработке удалось сохранить **1259 из них**.